# Exploring some basics from the MCS Zooms
## Written by Eric Rohr

In [ ]:
### import modules
import illustris_python as il # type: ignore
import matplotlib.pyplot as plt 
import numpy as np 
import matplotlib as mpl 
import matplotlib.cm as cm 
import matplotlib.patheffects as pe 
import matplotlib.transforms as transforms  
from matplotlib.gridspec import GridSpec  
import matplotlib.gridspec as gridspec  
from matplotlib.patches import Patch  
import matplotlib.patches as patches  
from mpl_toolkits.axes_grid1.inset_locator import inset_axes  
from mpl_toolkits.axes_grid1 import make_axes_locatable  
from scipy.ndimage import gaussian_filter  
from scipy import ndimage  
from scipy.interpolate import interp1d  
from scipy import interpolate  
from temet.util.sphMap import sphMap  #type: ignore
import scipy.stats  
from scipy.stats import norm  
from sklearn.neighbors import KernelDensity  
from scipy.stats import ks_2samp, anderson_ksamp  
from scipy.optimize import curve_fit  
from astropy.cosmology import Cosmology, FlatLambdaCDM, z_at_value
from astropy import units as u 
from astropy import constants as const 
import os
import glob
import csv
from pathlib import Path
import time
import h5py  
import rohr_utils as ru 
import utils.io as io
from utils.units import *
import random
import six  
import scida
from scida import load
import pint 
from createMCSTFiles import createMCSTFiles
import createOffsets
from stellar_array_helpers import expand_all_arrays

%matplotlib inline

plt.style.use('fullpage.mplstyle')

os.chdir('/u/reric/Scripts/')
! pwd



In [ ]:
def prepareSim(simFamily, simName, globalStartPath='/virgotng/universe', localSimFamilyPath='../'):
    """
    Prepare the given simulation for analysis. Returns the basePath.
    """
    basePath = createSimDirecStruct(simFamily, simName, globalStartPath=globalStartPath, localSimFamilyPath=localSimFamilyPath)
    io.createSnapTimes(basePath)
    snapTimes = io.loadSnapTimes(basePath)
    for snapNum in snapTimes['SnapNum']:
        createOffsets.createOffsets(basePath, snapNum)

    return basePath


def createSimDirecStruct(simFamily, simName, globalStartPath='/virgotng/universe', localSimFamilyPath='../'):
    """
    create a local copy using symbolic links to a given simulation.
    the default global path is /virgotng/universe, which then gets
    combined with simFamily (and output) to become the global basePath.
    localSimFamilyPath, which defaults to the parent directory, will 
    then hold the simFamily directory, which holds simName. Lastly, 
    local basePath = localSimFamilyPath + simFamily + simName + output.
    if local simFamily directory already exists, then nothing is done.
    Returns the local basePath.
    """

    localSimPath = os.path.join(localSimFamilyPath, 'sims.' + simFamily, simName)
    localbasePath = os.path.join(localSimPath, 'output')
    if os.path.isdir(localSimPath):
        return localbasePath
    else:
        os.makedirs(localSimPath)

    # output directory should be a sym link to the global direc
    globalBasePath = os.path.join(globalStartPath, simFamily, simName, 'output')
    os.symlink(globalBasePath, localbasePath, target_is_directory=True)

    # postprocesing directory should be local, but existing catalogs should be linked
    localPostprocessingPath = os.path.join(localSimPath, 'postprocessing')
    os.makedirs(localPostprocessingPath)

    globalPostprocessingPath = os.path.join(Path(globalBasePath).parent, 'postprocessing')
    os.system('ln -s %s %s'%(os.path.join(globalPostprocessingPath, '*'), os.path.join(localPostprocessingPath, '.')))

    return localbasePath

    

In [ ]:
class Sim: 
    """Create a class for the given simulation"""

    def __init__(self, simFamily, simName):
        """ initialize the class with basic info from the simulation"""

        kwargs = locals().copy()
        for _key in kwargs:
            if _key != 'self':
                setattr(self, _key, kwargs[_key])

        self.basePath = prepareSim(self.simFamily, self.simName)
        self.snapTimes = io.loadSnapTimes(self.basePath)
        self.Header = io.loadHeader(self.basePath, self.snapTimes['SnapNum'][0])
        self.Parameters = io.loadParameters(self.basePath, self.snapTimes['SnapNum'][0])
        self.Config = io.loadConfig(self.basePath, self.snapTimes['SnapNum'][0])
        self.snapNum_z0 = io.findSnapNum(self.basePath, 'Redshift', 0.0)

        # define short titles for plotting
        add_kwargs(self)


plot_kwargs = dict(marker='o', fillstyle='none', ms=3, mew=1.0, alpha=0.5)
med_kwargs = dict(marker='None', ls='-', lw=3, path_effects=[pe.Stroke(linewidth=4, foreground='white'), pe.Normal()], zorder=3)
hist_kwargs = dict(lw=0.2, alpha=0.4, ls='-')
percentiles_kwargs = dict(alpha=0.2)

def add_kwargs(sim, **kwargs):
    """ add kwargs to class"""

    if (sim.simFamily == 'IllustrisTNG'):
        c = 'tab:blue'
        label =  sim.simName[:-2]
    elif sim.simFamily == 'Eagle':
        c = 'fuchsia'
        label = 'Eagle'
    elif sim.simFamily == 'Simba':
        c = 'tab:orange'
        label = 'Simba'
    elif sim.simFamily == 'Illustris':
        c = 'tab:olive'
        label = 'Illustris'

    kwargs['c'] = kwargs['color'] = c
    kwargs['label'] = label

    sim.kwargs = kwargs
    return



In [ ]:
inputs = dict(IllustrisTNG=['TNG100-1'],
              Eagle=['Eagle100-1'],
              Illustris=['Illustris-1'],
              Simba=['Simba100-1'])

Sims = {}

for simFamily in inputs:
    simNames = inputs[simFamily]
    for simName in simNames:
        print(simFamily, simName)
        Sims[simName] = Sim(simFamily, simName)

In [ ]:
Sims['TNG100-1'].snapTimes['SnapNum']

In [ ]:
halo_fields = ['Group_M_Crit200', 'GroupMassType', 'GroupFirstSub']
subhalo_fields = ['SubhaloMassInRadType', 'SubhaloSFRinHalfRad', 'SubhaloGrNr']
for simName in Sims:
    sim = Sims[simName]
    sim.Halos_z0 = il.groupcat.loadHalos(sim.basePath, sim.snapNum_z0, halo_fields)
    io.convertGroupUnits(sim.basePath, sim.snapNum_z0, sim.Halos_z0)
    maskHalosCentrals_z0 = sim.Halos_z0['GroupFirstSub'] >= 0
    Subhalos = il.groupcat.loadSubhalos(sim.basePath, sim.snapNum_z0, subhalo_fields)
    r = {}
    for key in subhalo_fields:
        r[key] = Subhalos[key][(sim.Halos_z0['GroupFirstSub'][maskHalosCentrals_z0]).astype(np.int32)]
    r['count'] = sim.Halos_z0['GroupFirstSub'][maskHalosCentrals_z0].size
    io.convertGroupUnits(sim.basePath, sim.snapNum_z0, r)
    sim.Centrals_z0 = r
    sim.maskHalosCentals_z0 = maskHalosCentrals_z0

In [ ]:
savefig = False
outdirec = '../Figures/CosmoSimComparisons'
if not os.path.isdir(outdirec):
    os.makedirs(outdirec)

In [ ]:
# let's plot Mstar vs M200c at z=0 for all central galaxies and their host halos

def smoothCurve(ar, type='Gaussian', type_kwargs=dict(sigma=1)):
    """
    Smooth the given array by interpolating and applying a type of filter.
    Currently only supported for type == 'Gaussian', with a default sigma=1.
    """

    if not isinstance(ar, np.ndarray):
        ar = np.array(ar)
        if ar.size <= 1:
            return
    
    if type == 'Gaussian':
        return gaussian_filter(ar, **type_kwargs)
    else:
        raise ValueError('type %s not currently supported.'%type)
    

fig, ax = plt.subplots()

x_min = 10.**(10.5) * u.M_sun
binwidth = 0.2 # dex
flag_smoothCurve = True

for sim_i, simName in enumerate(Sims):
    
    sim = Sims[simName]
    add_kwargs(sim)
    _x = sim.Halos_z0['Group_M_Crit200'][sim.maskHalosCentals_z0]
    x_mask = _x > x_min
    x = _x[x_mask]
    _y = sim.Centrals_z0['SubhaloMassInRadType'][x_mask,4] / x
    y_mask = _y > 0
    y = _y[y_mask]
    x = x[y_mask]

    r = ru.return2dhiststats_dict(np.log10(x.value), np.log10(y.value), binwidth, percentiles=[16, 50, 84])
    if flag_smoothCurve:
        for key in r:
            r[key] = smoothCurve(r[key])

    #ax.plot(x[mask], y[mask] / y[mask], marker='.', ls='None', ms=1, alpha=0.2)
    ax.plot(10.**(r['bin_cents']), 10.**(r[50]), c=sim.kwargs['c'], **med_kwargs, label=sim.kwargs['label'] + ' (%d)'%(x.size))
    ax.fill_between(10.**(r['bin_cents']), 10.**(r[16]), 10.**(r[84]), color=sim.kwargs['c'], **percentiles_kwargs)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(10.**(10.5), 10.**(14.5))
ax.legend(title=r'Centrals at $z=0$')
ax.set_xlabel(r'Halo Mass $[M_{\rm 200c} / \rm{M_\odot}]$')
ax.set_ylabel(r'Stellar to Halo Mass Ratio $[M_\star / M_{\rm 200c}]$')

if savefig:
    fname = 'SHMR_z0.pdf'
    fig.savefig(os.path.join(outdirec, fname), bbox_inches='tight')

## begin re-developing createSGRP and associated codes to run for all cosmo sims

In [ ]:
sim = Sim('IllustrisTNG', 'TNG100-3')

In [ ]:
### initialize subfindIDs of interest
m200c_lolim = 10**(10.5) * u.M_sun
mstar_lolim = 0 * u.M_sun
halo_fields = ['Group_M_Crit200', 'GroupFirstSub']
subhalo_fields = ['SubhaloMassInRadType', 'SubhaloGrNr']

Halos = il.groupcat.loadHalos(sim.basePath, sim.snapNum_z0, halo_fields)
io.convertGroupUnits(sim.basePath, sim.snapNum_z0, Halos)

maskHalosCentrals = (Halos['Group_M_Crit200'] > m200c_lolim) & (Halos['GroupFirstSub'] >= 0)
GroupFirstSub = Halos['GroupFirstSub'][maskHalosCentrals].astype(int)

Subhalos = il.groupcat.loadSubhalos(sim.basePath, sim.snapNum_z0, subhalo_fields)
io.convertGroupUnits(sim.basePath, sim.snapNum_z0, Subhalos)
for key in subhalo_fields:
    Subhalos[key] = Subhalos[key][GroupFirstSub]

maskMstar = Subhalos['SubhaloMassInRadType'][:,4] > mstar_lolim

subfindIDs = GroupFirstSub[maskMstar]


In [ ]:
subfindIDs.value

In [ ]:
snapTimes = sim.snapTimes
min_redshift = 0.
max_redshift = 2.

start_snap = io.findSnapNum(sim.basePath, 'Redshift', min_redshift)
stop_snap = io.findSnapNum(sim.basePath, 'Redshift', max_redshift)

N_snaps = start_snap - stop_snap + 1
all_snaps = np.arange(start_snap, stop_snap-1, - 1, dtype=int)

keys = ['SnapNum', 'SubfindID']

scalar_template = np.zeros((subfindIDs.size, N_snaps), dtype=float) - 1
r = {}
for key in keys:
    r[key] = scalar_template.copy()

for subfindID_i, subfindID in enumerate(subfindIDs):
    tree = io.loadMainTreeBranch(sim.basePath, sim.snapNum_z0, subfindID, treeName='SubLink', stop_snap=stop_snap, start_snap=start_snap)
    indices = ru.find_common_snaps(all_snaps, tree['SnapNum'], tree['SnapNum'])[0]
    for key in keys:
        if key == 'SnapNum':
            r[key][subfindID_i,:] = all_snaps
        else:
            r[key][subfindID_i,indices] = tree[key] 


In [ ]:
r['SnapNum'].astype(int)[0], r['SubfindID'].astype(int)[0]

In [ ]:
sim.basePath
GRPdirec = os.path.join(Path(sim.basePath).parent, 'postprocessing', 'subfindGRP')
if not os.path.isdir(GRPdirec):
    os.system('mkdir %s'%GRPdirec)

GRPfname = 'CosmoSimComparison_subfind.hdf5'
fullGRPfname = os.path.join(GRPdirec, GRPfname)


In [ ]:
with h5py.File(fullGRPfname, 'w') as f:
    for key in r:
        f.create_dataset(key, data=r[key])
    f.close()